# HRS DDL and DML Template Architecture

---

## 1. Document Information

| Property           | Value                                                      |
| --- | --- |
| Document Name      | HRS HRS DDL and DML Template Architecture                  |
| Version            | 1.0                                                        |
| Author             | Perez                                                      |
| Last Updated       | 2026-09-08                                                 |

---

# 2. Objective

Create one standardized DDL/DML-generation templates to support HRS survey ETL applications.  
1. Use a separate DML specification for each HRS survey section. 
2. The DML can reference the already-defined target DDL specification rather than repeatedly redefining it.

---

# 3. Scope
1. DDL Generation
2. DML Generation

# 4. Architecture
There are two parts to this Architecture.

1. HRS DDL Specification: `What does the table look like?`

* Template File: HRS_DML_Master_Template.ipynb
        
2. HRS DML Specification: `How do we populate it correctly from RAND HRS?`

* Template File: `HRS_DML_Master_Template.ipynb`

3. Process Flow

        HRS DDL SPECIFICATION
                 │
                 ▼
      Creates Silver CDM Table
                 │
 ┌───────────────┼───────────────┐
 ▼               ▼               ▼
Target Schema  Constraints   Data Types
 │               │               │
 └───────────────┼───────────────┘
                 ▼
            TARGET TABLE
                 │
                 ▼
       HRS DML SPECIFICATION
                 │
 ┌───────────────┼───────────────┐
 ▼               ▼               ▼
Lookups       Transform        Load
 │               │               │
 ▼               ▼               ▼
HHIDPN→ID     Unpivoting      INSERT
Wave→ID       Mappings        etc.


# DDL Generation Instructions
Generate a complete Databricks SQL DDL script from this specification.

## Template File: `HRS_DDL_Master_Template.ipynb`

The generated script does the following:

1. Create the specified managed Delta table.
1. Use the specified catalog, schema, and table name.
1. Create the system-generated identity primary key.
1. Define all required columns.
1. Apply the specified data types and nullability.
1. Define the respondent foreign key.
1. Define the wave foreign key.
1. Define applicable constraints.
1. Include table and column comments.
1. Follow the specified SQL formatting requirements.
1. Use only DDL statements.
1. Not generate INSERT, UPDATE, DELETE, MERGE, or other DML.
1. Not invent columns, constraints, transformations, or business rules that are not defined in this specification.
1. Produce the final SQL as the sole deliverable.

# DDL Generation Instructions
The DML specification as a companion to the master DDL specification, rather than duplicating the DDL information.

The specification fairly prescriptive because the difficult part is not the INSERT itself—it is correctly converting the wide RAND HRS longitudinal structure into your respondent-by-wave Silver CDM structure.  This will include pivoting many of the survey reponse variables from a flat horizontal structure to a vertical one.

## Template File: `HRS_DML_Master_Template.ipynb`

## Document Information

Identify the DML specification and its relationship to the DDL specification.

For example:

* Document Name
* Version
* Author
* Last Updated
* Target Platform
* Compute
* Client Runtime
* SQL Dialect
* Specification Type: DML Only
* Related DDL Specification
## Objective

Clearly state that the specification defines how source RAND HRS data is transformed and loaded into an existing Silver CDM table.

Something like:

* The purpose of this specification is to define the SQL DML required to transform source RAND HRS data into the target Silver CDM table created by the corresponding DDL specification.

## Scope

This is where we explicitly define what DML does and does not do.

Included:

* Source extraction
* Source filtering
* Wave identification
* Respondent identification
* Parent-key resolution
* Unpivoting
* Source-to-target mapping
* Data-type conversion
* NULL handling
* Business-rule transformations
* Audit-column population
* Target loading
* Duplicate handling
* DML validation

Excluded:

* CREATE TABLE
* DROP TABLE
* Table structure
* Primary-key definitions
* Foreign-key definitions
* Table comments
* Column definitions

That keeps DDL and DML cleanly separated.

## Source and Target Parameters

Reference the parameters from the DDL specification, rather than redefine everything.

For example:

| Parameter	| Value
| --- | --- |
| SOURCE_TABLE_NAME	| dev_catalog.brz_raw_hrs.randhrs1992_2022v1
| TARGET_TABLE_NAME	| dev_catalog.slv_cdm_hrs.fact_demographics
| RESPONDENT_PARENT_TABL0E	| dev_catalog.slv_cdm_hrs.hub_respondent
| WAVE_PARENT_TABLE	| dev_catalog.slv_cdm_hrs.dim_wave

This gives the DML generator everything it needs without duplicating the physical-table specification.

## Target Business Grain

This should be one of the most important sections.

For the model:

* One target row represents one respondent for one survey wave.

Therefore:

* Business Grain = respondent_id + wave_id

Distinguish the natural identifiers from the surrogate keys:

* HHIDPN       → identifies respondent
* wave_number  → identifies wave

* HHIDPN + wave_number maps to respondent_id + wave_id

This distinction is critical to the DML.

## Parent-Key Resolution

Respondent Resolution

The DML must resolve:

Source.HHIDPN
      ↓
hub_respondent.HHIDPN
      ↓
hub_respondent.respondent_id
6.2 Wave Resolution

Likewise:

Source.wave_number
      ↓
dim_wave.wave_number
      ↓
dim_wave.wave_id

The DML should never manufacture respondent_id or wave_id.

They already exist in the parent tables.

This is one of the most important rules I'd put into the template:

Surrogate keys must always be obtained from their respective parent tables. They must never be generated, calculated, or inferred by the DML process.
